# Agentic RAG + MCP

**Week 12 · Session 3**

Last session the analyst found the *what* — Orbit's rating crashed in March and returns
spiked — and then said something honest:

> *"The data can't show the root cause. Release notes or support tickets would."*

Today we give it those. The agent gets **three kinds of knowledge**, and decides for itself
which to use:

| Source | What's in it | How the agent reaches it |
|---|---|---|
| **Documents** | policies, manuals, firmware release notes | vector search (RAG) |
| **Data** | six months of sales | read-only SQL |
| **The web** | anything public and current | a search tool |

Then we plug in **MCP servers** — a support-ticket system and the official filesystem
server — and the agent writes an incident brief to disk.

This is **agentic RAG**: retrieval stops being a fixed first step (Week 8) and becomes a
tool the agent picks, repeats and combines.

In [1]:
%pip install -q "langchain==1.4.2" "langchain-openai==1.6.2" "langgraph==1.2.11" "langchain-text-splitters==1.1.2" "langchain-mcp-adapters==0.3.2" "mcp>=1.19,<2" tavily-python python-dotenv pandas


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, sys, json, re, sqlite3, getpass
from pathlib import Path
from typing import Optional

try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True) or Path.cwd().parent / ".env")
except ImportError:
    pass
for var in ["OPENAI_API_KEY", "TAVILY_API_KEY"]:
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

import numpy as np
import pandas as pd
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Anaconda 'base' fix (same as Week 11). httpx2 2.12+ decompresses brotli with output_buffer_limit=... as a keyword.
# Anaconda's brotli 1.x exposes process() as a C function that rejects kwargs
# ("TypeError: process() takes no keyword arguments"). Wrap it so OpenAI
# Responses (which often arrive Content-Encoding: br) can be decoded.
try:
    import brotli as _brotli
    from httpx2._decoders import BrotliDecoder as _BrotliDecoder
    _probe = _brotli.Decompressor()
    try:
        _probe.process(b"", output_buffer_limit=1)
    except TypeError:
        _brotli_init = _BrotliDecoder.__init__
        def _brotli_init_compat(self, *args, **kwargs):
            _brotli_init(self, *args, **kwargs)
            _impl = self._decompress
            self._decompress = lambda data, output_buffer_limit=None, **_k: _impl(data)
        _BrotliDecoder.__init__ = _brotli_init_compat
except Exception:
    pass

MODEL = "openai:gpt-4.1-mini"
llm = init_chat_model(MODEL, temperature=0)
print("ready")

ready


---
# Part 1 — The knowledge base

Nimbus's internal documents: policies, product manuals, a safety bulletin, and firmware
release notes. All fictional, all short, so you can read them.

In [3]:
DOCS = {
    "warranty-policy": ("Warranty policy",
        "All Nimbus products carry a 12-month warranty from the date of purchase covering manufacturing "
        "defects. Physical damage, liquid damage beyond the product's IP rating, and battery wear below "
        "80% capacity after 12 months are not covered. Claims need the order ID."),
    "returns-policy": ("Returns and replacements",
        "Customers can return any product within 7 days of delivery for a full refund. Defective products "
        "are replaced within 10 working days. Nimbus arranges free courier pickup for all returns and "
        "replacements; customers must never be asked to pay for shipping a defective item."),
    "shipping-policy": ("Shipping times",
        "Metro cities: 1-3 days. Tier-2 cities: 3-5 days. North-east India, including Guwahati: 7-10 days "
        "because orders ship from the Bengaluru warehouse. A Guwahati warehouse is planned for Q4 2026."),
    "support-sla": ("Support response times",
        "Email support replies within 48 hours. The chat assistant escalates to a human when the customer "
        "types 'agent'. Safety-related complaints must be escalated to the product-safety team within 4 hours."),
    "orbit-fw-2.2": ("Orbit firmware 2.2 release notes (9 Feb 2026)",
        "Adds SpO2 trend graphs and new workout modes. Improves step-count accuracy. No known issues."),
    "orbit-fw-2.3": ("Orbit firmware 2.3 release notes (5 Mar 2026)",
        "New watch faces and 'smart charging' that charges faster to 80%. Known issues (added 18 Mar): on some "
        "units the watch can become hot during smart charging; the companion app may log users out daily and "
        "lose unsynced workout data; notifications can be delayed by several minutes."),
    "orbit-fw-2.3.1": ("Orbit firmware 2.3.1 release notes (10 Apr 2026)",
        "Fixes: smart-charging temperature regulation (the watch no longer heats up while charging); app "
        "session bug that logged users out; workout data sync. Pushed automatically to all watches over 48 hours."),
    "orbit-safety-bulletin": ("Safety bulletin: Orbit charging (20 Mar 2026)",
        "A small number of Orbit watches on firmware 2.3 may get hot during smart charging. Customers should use "
        "the supplied cable, avoid charging on soft surfaces, and unplug if the watch feels hot. Firmware 2.3.1 "
        "will fix this. Any report of burning smell or heat must be escalated to product safety within 4 hours."),
    "orbit-manual": ("Orbit user manual (excerpt)",
        "Battery life: up to 9 days typical, 5 days with always-on display. To reset, hold the side button for "
        "15 seconds. GPS lock can take up to 2 minutes outdoors. iPhone and Android supported."),
    "pulse-x1-manual": ("Pulse X1 user manual (excerpt)",
        "Battery: 5 hours with ANC on, 7 hours with ANC off; 30 hours total with the case. To re-pair, put both "
        "buds in the case and hold the case button for 10 seconds. IPX4 sweat resistance."),
    "pulse-x1-fw-1.5": ("Pulse X1 firmware 1.5 release notes (20 Apr 2026)",
        "Fixes the left earbud occasionally disconnecting during calls. Improves microphone noise suppression."),
    "echo-bar-manual": ("Echo Bar user manual (excerpt)",
        "IPX5 rating: protected against splashes and rain, NOT submersion. Bluetooth range up to 10 metres in open "
        "space; walls reduce range. Battery: 12 hours at medium volume. No aux input."),
}

documents = [Document(page_content=f"{title}\n\n{text}", metadata={"doc_id": doc_id, "title": title})
             for doc_id, (title, text) in DOCS.items()]
print(len(documents), "documents")

12 documents


Chunk them, embed them, and put them in a vector store — the Week 8 pipeline in four
lines. `InMemoryVectorStore` keeps this notebook dependency-free; in production you'd use
Chroma, pgvector, Pinecone or similar, with the same interface.

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=60)
chunks = splitter.split_documents(documents)

vector_store = InMemoryVectorStore.from_documents(chunks, OpenAIEmbeddings(model="text-embedding-3-small"))
print(f"{len(chunks)} chunks embedded")

for doc, score in vector_store.similarity_search_with_score("why does my watch get hot", k=3):
    print(f"  {score:.3f}  {doc.metadata['doc_id']}")

12 chunks embedded


  0.487  orbit-safety-bulletin
  0.405  orbit-fw-2.3
  0.355  orbit-fw-2.3.1


---
# Part 2 — Classic RAG, the v1 way

`RetrievalQA` is gone. A RAG chain in LCEL is: **retrieve → format → prompt → model**.

In [5]:
retriever = vector_store.as_retriever(search_kwargs={"k": 4})


def format_docs(docs):
    return "\n\n".join(f"[{d.metadata['doc_id']}] {d.page_content}" for d in docs)


rag_prompt = ChatPromptTemplate.from_template(
    "Answer using ONLY the context. Cite doc ids in [brackets]. If the context doesn't answer it, say so.\n\n"
    "Context:\n{context}\n\nQuestion: {question}"
)
classic_rag = ({"context": retriever | format_docs, "question": RunnablePassthrough()}
               | rag_prompt | llm | StrOutputParser())

print(classic_rag.invoke("What's the warranty on the Orbit, and does it cover water damage?"))

The Orbit has a 12-month warranty from the date of purchase covering manufacturing defects. However, liquid damage beyond the product's IP rating is not covered under the warranty [warranty-policy].


Good at what it's built for: one question, answered from documents. Now ask it something
that needs **data**:

In [6]:
print(classic_rag.invoke("Did the Orbit firmware 2.3 update hurt ratings? Show the average rating "
                         "before, during and after the problem."))

The context does not provide any information about Orbit firmware 2.3 update's impact on ratings or average ratings before, during, and after the problem. Therefore, this question cannot be answered from the given context.


It found the release notes and can tell you *what* 2.3 broke. It **cannot** give you
ratings: they're in a database, not in any document. Classic RAG always does exactly one
thing — retrieve, then answer — so it can't notice it needs a different source.

That's the limit agentic RAG removes.

---
# Part 3 — Knowledge sources as tools

## Documents, with a relevance score

The tool returns each chunk's **score**, so the agent can tell a strong match from a weak
one, and search again with better words when it needs to.

In [7]:
@tool
def search_docs(query: str, k: int = 4) -> str:
    """Search Nimbus's internal documents: warranty, returns, shipping and support policies,
    product manuals, firmware release notes and safety bulletins.

    Each result has a relevance score from 0 to 1. Below about 0.35 the match is weak:
    rephrase the query with more specific words and search again.

    Args:
        query: What to look for, in plain words, e.g. "Orbit charging overheating".
        k: Number of results, 1-6.
    """
    hits = vector_store.similarity_search_with_score(query, k=max(1, min(k, 6)))
    return "\n\n".join(f"[{d.metadata['doc_id']}] (score {s:.2f}) {d.page_content}" for d, s in hits)


print(search_docs.invoke({"query": "Orbit hot while charging"})[:600])

[orbit-safety-bulletin] (score 0.63) Safety bulletin: Orbit charging (20 Mar 2026)

A small number of Orbit watches on firmware 2.3 may get hot during smart charging. Customers should use the supplied cable, avoid charging on soft surfaces, and unplug if the watch feels hot. Firmware 2.3.1 will fix this. Any report of burning smell or heat must be escalated to product safety within 4 hours.

[orbit-fw-2.3] (score 0.55) Orbit firmware 2.3 release notes (5 Mar 2026)

New watch faces and 'smart charging' that charges faster to 80%. Known issues (added 18 Mar): on some units the watch can become h


## Data, through read-only SQL

Last session we built safe aggregation tools. The other common approach is
**text-to-SQL**: let the model write queries. It's flexible and it's dangerous, so the
guardrail has to be real. A regex that checks for `SELECT` is not a guardrail. A
**read-only database connection** is.

In [8]:
def make_sales(seed=12, n=2400):
    rng = np.random.default_rng(seed)
    products = {"Pulse X1": 2499, "Orbit": 3999, "Echo Bar": 1799}
    cities = ["Bengaluru", "Mumbai", "Delhi", "Chennai", "Pune", "Hyderabad", "Kolkata", "Guwahati"]
    city_w = [0.18, 0.18, 0.17, 0.12, 0.10, 0.11, 0.08, 0.06]
    base_delivery = {"Bengaluru": 2, "Mumbai": 2, "Delhi": 2, "Chennai": 3, "Pune": 3, "Hyderabad": 3,
                     "Kolkata": 5, "Guwahati": 9}
    channels = ["Website", "Amazon", "Flipkart", "Retail"]
    disc = {"Website": (0, 10), "Amazon": (5, 15), "Flipkart": (10, 25), "Retail": (0, 5)}
    dates = pd.to_datetime("2026-01-01") + pd.to_timedelta(rng.integers(0, 181, n), unit="D")
    rows = []
    for i, d in enumerate(sorted(dates)):
        p = rng.choice(list(products), p=[0.42, 0.28, 0.30])
        c = rng.choice(cities, p=city_w)
        ch = rng.choice(channels, p=[0.22, 0.34, 0.30, 0.14])
        lo, hi = disc[ch]
        dpct = int(rng.integers(lo, hi + 1))
        units = int(rng.choice([1, 1, 1, 1, 2]))
        dd = 0 if ch == "Retail" else max(1, int(base_delivery[c] + rng.integers(-1, 3)))
        hidden = p == "Orbit" and pd.Timestamp("2026-03-05") <= d < pd.Timestamp("2026-04-10")
        rating = int(np.clip(round(4.3 - (1.6 if hidden else 0) - 0.12 * max(0, dd - 3)
                                   + rng.normal(0, 0.55)), 1, 5))
        rows.append({"order_id": f"N{100000 + i}", "order_date": d.date().isoformat(), "product": p,
                     "city": c, "channel": ch, "units": units, "revenue_inr": round(units * products[p] * (1 - dpct / 100)),
                     "discount_pct": dpct, "delivery_days": dd, "rating": rating,
                     "returned": int(rng.random() < (0.22 if hidden else 0.05))})
    return pd.DataFrame(rows)


Path("sales.db").unlink(missing_ok=True)
with sqlite3.connect("sales.db") as con:
    make_sales().to_sql("orders", con, index=False)
print("sales.db written:", pd.read_sql("SELECT COUNT(*) AS n FROM orders", sqlite3.connect("sales.db")).n[0], "orders")

sales.db written: 2400 orders


In [9]:
@tool
def query_sales(sql: str) -> str:
    """Run a read-only SQL query (SQLite) against Nimbus sales data. Returns up to 40 rows.

    Table `orders` columns: order_id, order_date (TEXT 'YYYY-MM-DD'), product ('Pulse X1'|'Orbit'|'Echo Bar'),
    city, channel ('Website'|'Amazon'|'Flipkart'|'Retail'), units, revenue_inr, discount_pct,
    delivery_days, rating (1-5), returned (0/1). Data covers 2026-01-01 to 2026-06-30.
    Always include COUNT(*) AS n for any average or rate, so the sample size is visible.
    Write rates as percentages, e.g. ROUND(100.0 * AVG(returned), 1) AS return_rate_pct --
    never report AVG(returned) as a raw fraction.
    """
    try:
        # mode=ro: the connection itself cannot write -- that's the real guardrail
        with sqlite3.connect("file:sales.db?mode=ro", uri=True) as con:
            result = pd.read_sql(sql, con)
    except Exception as exc:
        return f"SQL error: {exc}. Check column names against the schema in this tool's description."
    more = f"\n... {len(result) - 40} more rows" if len(result) > 40 else ""
    return result.head(40).to_string(index=False) + more


print(query_sales.invoke({"sql": "SELECT product, ROUND(AVG(rating),2) AS avg_rating, COUNT(*) AS n "
                                 "FROM orders GROUP BY product"}))
print(query_sales.invoke({"sql": "DELETE FROM orders"}))          # try to destroy the data

 product  avg_rating   n
Echo Bar        4.21 729
   Orbit        3.93 678
Pulse X1        4.24 993
SQL error: Execution failed on sql 'DELETE FROM orders': attempt to write a readonly database. Check column names against the schema in this tool's description.


The `DELETE` failed at the **database**, not in our code: *"attempt to write a readonly
database"*. The model can write any SQL it likes; the connection simply can't change
anything. Guardrails belong at the layer that can actually enforce them.

## The web

In [10]:
from tavily import TavilyClient

_tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])


@tool
def web_search(query: str) -> str:
    """Search the public web for current, external information: industry news, standards,
    competitors, regulations. NOT for anything about Nimbus's own products, policies or sales --
    use search_docs or query_sales for those."""
    res = _tavily.search(query=query, max_results=3)
    return "\n\n".join(f"- {r['title']} ({r['url']})\n  {r['content'][:300]}" for r in res["results"])


print(web_search.invoke({"query": "smartwatch charging overheating safety standard India"})[:500])

- How to Fix Smartwatch Overheating While Charging? (https://store.blackview.hk/blogs/gadgets-guides/smartwatch-overheats-while-charging)
  Skip to content 

Worldwide Free Shipping - Shop Now!

Blackview Official StoreBlackview Official Store

#### Most searched products

Back and front of the yellow Blackview FORT 5 Ultra

Black Blackview FORT 5 Ultra, back and front. 

Sold out

### Blackview FORT 5 Ultra-slim Wireless Charging 6.78-

- Smart watch heating problem solved? l overheating fix fo


Read the three descriptions again. Each one says **what it's for and what it isn't** —
"NOT for anything about Nimbus's own products". That's how the agent routes: it reads
the descriptions. Routing quality is description quality.

---
# Part 4 — The agentic RAG agent

In [11]:
RAG_PROMPT = """You are Nimbus's product-intelligence assistant.

SOURCES — choose deliberately, and combine them when a question needs more than one
- search_docs: Nimbus policies, manuals, firmware release notes, safety bulletins.
- query_sales: order data for 2026-01-01 to 2026-06-30 (ratings, returns, revenue, delivery).
- web_search: external, public information only.

METHOD
- Break multi-part questions into sub-questions. Answer each from the right source.
- If a search_docs result scores below 0.35, rephrase and search again (at most twice).
- When a document dates a change (e.g. released 5 Mar, fixed 10 Apr), use THOSE EXACT dates as
  the window boundaries in SQL: before = order_date < first date; during = from the first date
  up to (not including) the second; after = on or after the second date. Don't pick your own.

OUTPUT
- Lead with the answer. Cite every fact: [doc-id] for documents, (SQL) for data, (web) for web.
- Every number must come from a tool result. Report sample sizes for averages and rates.
- If no source answers the question, say so plainly. Never guess."""

rag_agent = create_agent(MODEL, tools=[search_docs, query_sales, web_search], system_prompt=RAG_PROMPT)


def run(question, agent=None):
    agent = agent or rag_agent            # look up rag_agent at CALL time, so a rebuilt agent is used
    r = agent.invoke({"messages": [{"role": "user", "content": question}]})
    used = [tc["name"] for m in r["messages"] for tc in (getattr(m, "tool_calls", None) or [])]
    print(f"Q: {question}\nsources used: {used}\n\n{r['messages'][-1].content}\n" + "-" * 90)
    return r

## Routing: four questions, four different source patterns

In [12]:
run("What's the return policy if my Echo Bar arrives damaged?")
run("What was Orbit's return rate in March 2026?")
run("What's the latest Bluetooth version most budget earbuds ship with in 2026?")
run("What is Nimbus's revenue target for Q4 2026?")

Q: What's the return policy if my Echo Bar arrives damaged?
sources used: ['search_docs']

If your Echo Bar arrives damaged, you can return it within 7 days of delivery for a full refund. Nimbus will arrange free courier pickup for the return, and you should not be asked to pay for shipping. Defective products are replaced within 10 working days. You will need the order ID to make a claim. This return and replacement policy covers damaged or defective products upon arrival [returns-policy], [warranty-policy].
------------------------------------------------------------------------------------------


Q: What was Orbit's return rate in March 2026?
sources used: ['query_sales']

Orbit's return rate in March 2026 was 14.8%, based on 108 orders. (SQL)
------------------------------------------------------------------------------------------


Q: What's the latest Bluetooth version most budget earbuds ship with in 2026?
sources used: ['web_search', 'web_search']

The latest Bluetooth version that most budget earbuds ship with in 2026 is Bluetooth 5.3. This is common among widely recommended budget models such as the Soundcore Liberty 4 NC and EarFun Air Pro 4+, which are noted for their feature sets including multipoint Bluetooth connectivity typically supported by Bluetooth 5.3 or later versions [web].
------------------------------------------------------------------------------------------


Q: What is Nimbus's revenue target for Q4 2026?
sources used: []

Nimbus's revenue target for Q4 2026 is not available in the internal documents or sales data covering up to Q2 2026. There is no direct information on the revenue target for Q4 2026 in the sources I have access to.
------------------------------------------------------------------------------------------


{'messages': [HumanMessage(content="What is Nimbus's revenue target for Q4 2026?", additional_kwargs={}, response_metadata={}, id='342241cf-7e86-4396-9b9f-5d894bb69b72'),
  AIMessage(content="Nimbus's revenue target for Q4 2026 is not available in the internal documents or sales data covering up to Q2 2026. There is no direct information on the revenue target for Q4 2026 in the sources I have access to.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 658, 'total_tokens': 711, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_6f31bfbb14', 'id': 'chatcmpl-EPp7yPD6kX2ljxfvImaTCHIDy8FDp

Four different decisions: documents only; data only; web only; and **nothing** — a good
agent says it doesn't know rather than inventing a target. That last one is the question
classic RAG handles worst, because it always retrieves *something* and then tries to
answer from it.

## Multi-step: the question from last session

In [13]:
firmware = run("Did Orbit firmware 2.3 cause the rating drop? Compare average rating and return rate "
               "before 2.3, while 2.3 was live, and after 2.3.1 — and explain what went wrong.")

Q: Did Orbit firmware 2.3 cause the rating drop? Compare average rating and return rate before 2.3, while 2.3 was live, and after 2.3.1 — and explain what went wrong.
sources used: ['search_docs', 'search_docs', 'query_sales']

Orbit firmware 2.3 was released on 5 March 2026, and it introduced new watch faces and a "smart charging" feature that charges faster to 80%. However, known issues surfaced by 18 March 2026 affecting some units: the watch could become hot during smart charging, the companion app might log users out daily losing unsynced workout data, and notifications could be delayed by several minutes. A safety bulletin on 20 March 2026 warned users about the heating issue and recommended precautions. Firmware 2.3.1, released on 10 April 2026, fixed the heating issue, app session logouts, and workout data sync problems [orbit-fw-2.3], [orbit-fw-2.3.1], [orbit-safety-bulletin].

Sales data comparison for the "Orbit" product in the first half of 2026 shows:
- Before 2.3 (before 

This is the payoff.

> **Two fixes we needed first.** Our first runs picked their *own* window boundaries (1 March,
> 14 April — anything but the documented dates), and once reported a 13% return rate as
> "around 3–5%", misreading the raw fraction `0.133`. Session 2's lesson again: the prompt now
> pins the windows to the documented dates, and the SQL tool asks for rates as percentages.

The agent should:

1. search the docs and find 2.3 shipped on **5 March** and 2.3.1 fixed it on **10 April**;
2. use **those exact dates** to query the data for three windows;
3. put the two together — ratings and returns collapse inside the 2.3 window and recover
   after 2.3.1 — and cite the release notes for *what* broke (hot while charging, logouts).

Neither source could answer this alone. The documents don't have the numbers, and the data
doesn't know about firmware. **Combining sources is the point of agentic RAG.**

### Don't trust the answer text — check the trajectory

A fluent answer can say "2.3 was released on 5 March" while its SQL quietly used 10 March.
The text and the query disagree, and only the trajectory shows it. So check it:

In [14]:
DOCUMENTED = ["2026-03-05", "2026-04-10"]       # from orbit-fw-2.3 and orbit-fw-2.3.1


def check_windows(result):
    """Show each step's tool calls, then compare the dates the SQL used with the documents."""
    used = set()
    ai_steps = [m for m in result["messages"] if m.type == "ai" and m.tool_calls]
    for i, m in enumerate(ai_steps, 1):
        print(f"step {i}: {[tc['name'] for tc in m.tool_calls]}")
        for tc in m.tool_calls:
            if tc["name"] == "query_sales":
                used |= set(re.findall(r"2026-\d\d-\d\d", tc["args"]["sql"]))
    print("\ndates the release notes give:", DOCUMENTED)
    print("dates the agent's SQL used  :", sorted(used))
    if set(DOCUMENTED) <= used:
        print("\nOK: the windows match the documents.")
    else:
        print("\nMISMATCH: the numbers describe different windows than the answer claims.\n"
              "The text may still say '5 March'.")


check_windows(firmware)

step 1: ['search_docs', 'search_docs', 'query_sales']

dates the release notes give: ['2026-03-05', '2026-04-10']
dates the agent's SQL used  : ['2026-03-05', '2026-04-10']

OK: the windows match the documents.


### Why the dates go wrong: parallel tool calls

In our test runs, `gpt-4.1-mini` wrote the wrong windows **four times out of seven**: 10
March, 15 April, once 1 January. Each time, the answer text still quoted "5 March" from the
release notes. And the instruction to use the documented dates was right there in the
prompt.

Look at **step 1** in the trajectory above whenever it fails: `search_docs` *and*
`query_sales` in the **same step**. Parallel tool calls are requested together, so the SQL
was written *before* the release notes came back. The model hadn't read the dates yet, so it
guessed them.

Last week we celebrated parallel calls. Here they bite: **calls in the same step can't
depend on each other.** When step B needs step A's answer, turn parallel calls off.

In [15]:
from langchain_openai import ChatOpenAI

# One step at a time: read the documents, THEN write the query.
sequential_llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0, model_kwargs={"parallel_tool_calls": False})

rag_agent = create_agent(sequential_llm, tools=[search_docs, query_sales, web_search], system_prompt=RAG_PROMPT)
firmware = run("Did Orbit firmware 2.3 cause the rating drop? Compare average rating and return rate "
               "before 2.3, while 2.3 was live, and after 2.3.1 — and explain what went wrong.")
check_windows(firmware)

Q: Did Orbit firmware 2.3 cause the rating drop? Compare average rating and return rate before 2.3, while 2.3 was live, and after 2.3.1 — and explain what went wrong.
sources used: ['search_docs', 'query_sales']

The Orbit firmware 2.3 release on 5 March 2026 coincided with a significant drop in average product rating and a spike in return rate. Before 2.3, the average rating was 4.18 (sample size 233) with a return rate of 5.2%. During the period when 2.3 was live (5 March to 9 April 2026), the average rating dropped sharply to 2.70 (sample size 122) and the return rate increased to 16.4%. After the 2.3.1 update on 10 April 2026, which fixed overheating during smart charging and app logout bugs, the average rating recovered to 4.22 (sample size 323) and the return rate dropped back to 3.4% [orbit-fw-2.3, orbit-fw-2.3.1, orbit-safety-bulletin] (SQL).

The issues causing the rating drop and returns during 2.3 were overheating during smart charging, app session logouts, loss of unsynced 

Now `search_docs` runs alone in step 1, and step 2's SQL uses the dates it just read. In our
tests that took us from **3 in 7** correct to **4 in 4**, with no change to the prompt or
the tools.

The trade-off is speed: more steps, each waiting for the last. So it's a per-agent choice.
Keep parallel calls when the calls are independent (Week 11's "flights, hotel and visa" in
one go); turn them off when later calls depend on earlier results.

Two principles from this section:

1. **A check you run on every answer beats an instruction the model follows most of the
   time.** `check_windows` is Week 10's trajectory evaluation in a dozen lines, and it's
   how we found the real cause.
2. **Read the trajectory, not just the answer.** The failing answers were fluent, cited the
   right document — and described the wrong numbers.

---
# Part 5 — Did it route correctly? Measure it.

Week 10's lesson applies here too: don't trust routing by eye. Here's a small trajectory
eval — for each question, which source *should* be used.

In [16]:
ROUTING_EVAL = [
    ("Is liquid damage covered by the warranty?",                     {"search_docs"}),
    ("How many Pulse X1 orders came from Pune?",                       {"query_sales"}),
    ("What does firmware 2.3.1 fix on the Orbit?",                     {"search_docs"}),
    ("Why is delivery to Guwahati slow, and how slow is it really?",    {"search_docs", "query_sales"}),
    ("Which channel gives the highest average discount?",             {"query_sales"}),
    ("What IP rating does the Echo Bar have and what does it mean?",   {"search_docs"}),
]

rows = []
for question, expected in ROUTING_EVAL:
    r = rag_agent.invoke({"messages": [{"role": "user", "content": question}]})
    used = {tc["name"] for m in r["messages"] for tc in (getattr(m, "tool_calls", None) or [])}
    rows.append({"question": question[:55], "expected": sorted(expected), "used": sorted(used),
                 "correct": expected <= used, "extra": sorted(used - expected)})

report = pd.DataFrame(rows)
print(report.to_string(index=False))
print(f"\nrouting accuracy: {report['correct'].mean():.0%}")

                                               question                   expected                       used  correct extra
              Is liquid damage covered by the warranty?              [search_docs]              [search_docs]     True    []
               How many Pulse X1 orders came from Pune?              [query_sales]              [query_sales]     True    []
             What does firmware 2.3.1 fix on the Orbit?              [search_docs]              [search_docs]     True    []
Why is delivery to Guwahati slow, and how slow is it re [query_sales, search_docs] [query_sales, search_docs]     True    []
      Which channel gives the highest average discount?              [query_sales]              [query_sales]     True    []
What IP rating does the Echo Bar have and what does it               [search_docs]              [search_docs]     True    []

routing accuracy: 100%


`correct` means it used every source it needed. `extra` shows sources it didn't need —
not wrong, but each costs a call. That's Week 10's trajectory evaluation, applied to
routing. Change a tool description and rerun this cell to see whether routing got better
or worse.

---
# Part 6 — MCP servers, through LangChain

Two weeks ago you built MCP servers. Last week you plugged one into the Agents SDK. Now
LangChain: **`langchain-mcp-adapters`** turns any MCP server's tools into ordinary
LangChain tools.

We'll connect two servers at once:

1. **A Nimbus support-desk server** — we write it now, with FastMCP.
2. **The official filesystem server** — off the shelf, via `npx` (skipped if you don't have Node).

In [17]:
%%writefile nimbus_support_server.py
"""Nimbus support desk -- a small FastMCP server over a tickets table."""
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("nimbus-support")

TICKETS = [
    {"id": "T-2101", "product": "Orbit",    "opened": "2026-03-09", "status": "open",   "summary": "Watch hot while charging, burning smell"},
    {"id": "T-2102", "product": "Orbit",    "opened": "2026-03-11", "status": "open",   "summary": "App logs me out every day, lost workouts"},
    {"id": "T-2103", "product": "Orbit",    "opened": "2026-03-14", "status": "closed", "summary": "Strap broke, replaced"},
    {"id": "T-2104", "product": "Orbit",    "opened": "2026-03-22", "status": "open",   "summary": "Notifications arrive 5 minutes late"},
    {"id": "T-2105", "product": "Pulse X1", "opened": "2026-03-02", "status": "open",   "summary": "Left bud disconnects during calls"},
    {"id": "T-2106", "product": "Echo Bar", "opened": "2026-02-18", "status": "closed", "summary": "Stopped working after rain"},
]


@mcp.tool()
def list_tickets(product: str, status: str = "open") -> str:
    """List support tickets for a product.

    Args:
        product: "Orbit", "Pulse X1" or "Echo Bar".
        status: "open", "closed" or "all".
    """
    hits = [t for t in TICKETS if t["product"].lower() == product.lower()
            and (status == "all" or t["status"] == status)]
    return "\n".join(f"{t['id']} [{t['status']}] {t['opened']}: {t['summary']}" for t in hits) or "No tickets."


if __name__ == "__main__":
    mcp.run(transport="stdio")

Overwriting nimbus_support_server.py


In [18]:
import shutil
from langchain_mcp_adapters.client import MultiServerMCPClient

REPORTS = Path("reports").resolve()
REPORTS.mkdir(exist_ok=True)

servers = {"support": {"command": sys.executable, "args": [str(Path("nimbus_support_server.py").resolve())],
                       "transport": "stdio"}}
if shutil.which("npx"):
    servers["files"] = {"command": shutil.which("npx"),
                        "args": ["-y", "@modelcontextprotocol/server-filesystem", str(REPORTS)],
                        "transport": "stdio"}
else:
    print("npx not found -- skipping the filesystem server (install Node to try it)")

client = MultiServerMCPClient(servers)
mcp_tools = await client.get_tools()
print(f"{len(mcp_tools)} MCP tools:", [t.name for t in mcp_tools])

15 MCP tools: ['list_tickets', 'read_file', 'read_text_file', 'read_media_file', 'read_multiple_files', 'write_file', 'edit_file', 'create_directory', 'list_directory', 'list_directory_with_sizes', 'directory_tree', 'move_file', 'search_files', 'get_file_info', 'list_allowed_directories']


The filesystem server alone brings 14 tools, including `move_file` and `edit_file`. Our
agent needs to *write one report*. Week 10's MCP security slide: **least privilege**. Give
the agent only the tools the job needs.

In [19]:
NEEDED = {"list_tickets", "write_file", "list_allowed_directories"}
mcp_tools = [t for t in mcp_tools if t.name in NEEDED]
print("kept:", [t.name for t in mcp_tools])

kept: ['list_tickets', 'write_file', 'list_allowed_directories']


Two things we learned wiring this up:

- **MCP tools are async.** Use `await agent.ainvoke(...)`. A plain `invoke` fails with
  *"StructuredTool does not support sync invocation"*.
- **The model doesn't know the server's root folder.** Our first run wrote to
  `reports/reports/…`: it prefixed a folder that was already the root. Tell the agent the
  absolute path it's allowed to write to.

---
# Part 7 — Capstone: the incident brief

Documents, data, tickets and a file write, in one request.

In [20]:
BRIEF_PROMPT = RAG_PROMPT + f"""

TOOLS FROM MCP
- list_tickets: Nimbus support tickets.
- write_file: write files. The ONLY folder you may write to is {REPORTS} — always use a full
  absolute path inside it."""

brief_agent = create_agent(sequential_llm, tools=[search_docs, query_sales] + mcp_tools,
                           system_prompt=BRIEF_PROMPT)

r = await brief_agent.ainvoke({"messages": [{"role": "user", "content":
    "Write an incident brief on Orbit's March problem to orbit_march_brief.md: what happened (docs), "
    "the impact on rating and returns before/during/after firmware 2.3 (data, with sample sizes), "
    "related support tickets, and which tickets are safety issues that breach our 4-hour escalation rule."}]})

for m in r["messages"]:
    for tc in (getattr(m, "tool_calls", None) or []):
        print(f"  ACTION  {tc['name']}({json.dumps(tc['args'])[:100]})")
print("\n" + r["messages"][-1].content)

  ACTION  search_docs({"query": "Orbit March problem firmware 2.3"})
  ACTION  query_sales({"sql": "SELECT\n  CASE\n    WHEN order_date < '2026-03-05' THEN 'before'\n    WHEN order_date >= '2)
  ACTION  list_tickets({"product": "Orbit", "status": "all"})
  ACTION  search_docs({"query": "Orbit charging safety escalation 4-hour"})
  ACTION  write_file({"path": "/Users/divijbajaj/Library/CloudStorage/OneDrive-Personal/AI_Agents_IndustryCaseStudy/Agent)

The incident brief on Orbit's March 2026 problem is written and saved to orbit_march_brief.md. It covers what happened (firmware 2.3 issues including heating during charging, app logout, notification delays), the impact on ratings and returns before/during/after firmware 2.3 with sample sizes (ratings dropped from 4.18 to 2.70 and returns rose from 5.2% to 16.4% during the issue period), related support tickets, and identifies the safety issue ticket (T-2101) that breaches the 4-hour escalation rule per the safety bulletin [orbit-fw-2.3, o

In [21]:
brief = next(REPORTS.rglob("orbit_march_brief.md"), None)
print(f"written to: {brief}\n" + "=" * 70)
print(brief.read_text() if brief else "no file written (did the filesystem server start?)")

written to: /Users/divijbajaj/Library/CloudStorage/OneDrive-Personal/AI_Agents_IndustryCaseStudy/Agentic_AI_Deck/week12/notebooks/reports/orbit_march_brief.md
# Incident Brief: Orbit Firmware 2.3 Issues in March 2026

## What Happened
On 5 March 2026, Nimbus released Orbit firmware version 2.3 which introduced new watch faces and a 'smart charging' feature that charges the watch faster to 80%. However, by 18 March, several issues were identified and documented:
- Some units of the Orbit watch became hot during smart charging.
- The companion app logged users out daily, causing loss of unsynced workout data.
- Notifications were delayed by several minutes.

A safety bulletin issued on 20 March 2026 highlighted the heating issue during charging, advising customers to use the supplied cable, avoid charging on soft surfaces, and unplug if the watch felt hot. It mandated that any report of burning smell or heat must be escalated to product safety within 4 hours. Firmware 2.3.1, released on 

That brief pulled from **four places**: release notes and the safety bulletin (RAG), rating
and return numbers for three windows (SQL), open tickets (MCP), and the escalation rule
from the support policy (RAG) — then wrote a file through a second MCP server.

It should also catch the uncomfortable part: ticket **T-2101** (hot while charging, burning
smell) is a safety issue, and the policy says safety complaints must be escalated within
4 hours. That's the kind of connection nobody makes when the data, the docs and the tickets
live in three different tools.

---
# Where Week 12 leaves you

| Session | You built | The lesson that stuck |
|---|---|---|
| 1 | a review pipeline, no agent | counting is code; most jobs are pipelines |
| 2 | an analyst agent + middleware | units, denominators; the model can't see its chart |
| 3 | agentic RAG + MCP | combine sources; guardrails at the layer that enforces them |

**Next week: LangGraph.** `create_agent` has been a prebuilt graph all week. Next week you
draw your own: explicit state, nodes, edges, loops and interrupts.

---
## Exercises

**1. Break routing.** Delete "NOT for anything about Nimbus's own products" from
`web_search`'s description and rerun the routing eval. Does accuracy drop?

**2. Corrective RAG.** Search for something vague ("the thing that got fixed in April").
Does the agent rephrase after a low score? Print its search queries.

**3. A 5th source.** Add a `list_tickets` question to the routing eval. Does the agent
reach for MCP correctly?

**4. SQL safety.** Ask the agent to "remove all test orders". What stops it? Which layer?

**5. Self-improving (stretch).** Log every question with the sources used and a thumbs-up or
thumbs-down. After ten questions, add the worst-routed examples to the system prompt as
few-shot guidance, and rerun the eval. Did routing improve?